# This notebook will run the baseline model provided by the competition. It can be found in the official git repository: https://github.com/royerlab/kaggle-cell-tracking-competition/tree/main

# Clonning repository


In [1]:
# # Clonning Git repository
# !git clone https://github.com/royerlab/kaggle-cell-tracking-competition.git
# %cd kaggle-cell-tracking-competition

In [2]:
# !ls #Elements in the git repository

In [3]:
# # -------------------------- Installing necessary dependencies (in case they are not installed) ----------------------------
# import importlib
# import subprocess, sys
# packages = [
#     "torch",
#     "zarr",
#     "polars",
#     "scipy",
#     "napari",
#     "tracksdata"
# ]
# for pkg in packages:
#     try:
#         module = importlib.import_module(pkg)
#         version = getattr(module, "__version__", "Unknown")
#         print(f"Already:  {pkg:<12} {version}")
#     except ImportError:
#         print(f" {pkg:<12} NOT INSTALLED")
#         print(f"Installing package: {pkg}:")
#         try:
#             subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
#             print(f"---- -- Package {pkg} successfully installed ------ --")
#         except ImportError:
#             print(f"---- --Package {pkg} was NOT installed ------ --")

# Defining input data

In [4]:
from pathlib import Path
import os

In [5]:
from pathlib import Path
import os
data_path = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development')
training_path =  data_path/'train'
test_path = data_path/'test'

#------------ Listing folders in train -----------
print("Input contents:", os.listdir(training_path)[:5],"\n")
print(" First 5 elements in train folder, it contains '.geff' folders (annotated graphs) and '.zarr' folder (videos)")

Input contents: ['6bba_2540cd90.geff', '44b6_0b24845f.geff', '44b6_996155de.geff', '44b6_0c582fdc.geff', '6bba_cf35214c.zarr'] 

 First 5 elements in train folder, it contains '.geff' folders (annotated graphs) and '.zarr' folder (videos)


# Making a prediction with the baseline model

The notebook used to run the baseline can be found here: https://www.kaggle.com/code/thibautgoldsborough/unet-baseline-inference-submission

## Configuration

In [6]:

COMP_DIR = data_path
TEST_DIR = test_path

ARTIFACTS = "/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts"

REPO_DIR = "/kaggle/working/repo"
METHOD = "unet_transformer"

# --- Test-time options (tune these) ---------------------------------------
# Model checkpoint (relative to the repo, or an absolute path to your own).
WEIGHTS = f"weights/{METHOD}/split_0/edge_predictor_best.pth"

# GT: Ground-Truth
# Detection peak threshold. GT is sparse so the detector is poorly calibrated;
# 0.5 is too low, ~0.99 scored best in a sweep.
DET_THRESHOLD = 0.99

UNET_BATCH_SIZE = 4          # frame-pairs per UNet forward; lower if you OOM
SLICE = ":4"                   # e.g. ":5" to predict only the first 5 videos; "" = all

# Linking. ILP = global, flow-consistent (cleaner tracks; ~0.73->0.79).
# Set False for the faster greedy linker.
USE_ILP = True
ILP_EDGE_WEIGHT = -1.0           # weight on edge probability
ILP_APPEARANCE_WEIGHT = 0.1      # cost of a track appearing
ILP_DISAPPEARANCE_WEIGHT = 0.1   # cost of a track disappearing
ILP_DIVISION_WEIGHT = 1.0        # cost of a division; lower (~0.2) allows more splits
# --------------------------------------------------------------------------

## Offline install & setup
Install the dependencies from the bundled wheels (no internet), then copy the repo source and weights into a writable location and put the package on the import path

In [7]:
import glob
import os
import subprocess
import sys
import shutil

# 1. Find all wheels, but filter OUT numpy wheels to avoid breaking C-extensions
wheels = [
    f for f in glob.glob(f"{ARTIFACTS}/wheels/*.whl")
    if "numpy" not in os.path.basename(f).lower()
]

# 2. Install only the required non-NumPy wheels without upgrading dependencies
subprocess.run(
    [
        "pip", "install", 
        "--no-index", 
        "--find-links", f"{ARTIFACTS}/wheels",
        "--no-deps",
        *wheels
    ],
    check=True,
)

# 3. Copy repo and weights as usual
shutil.copytree(f"{ARTIFACTS}/repo", REPO_DIR, dirs_exist_ok=True)
shutil.copytree(f"{ARTIFACTS}/weights", f"{REPO_DIR}/weights", dirs_exist_ok=True)
sys.path.insert(0, f"{REPO_DIR}/src")

Looking in links: /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/imagecodecs-2026.6.26-cp312-abi3-manylinux_2_28_x86_64.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/shellingham-1.5.4-py2.py3-none-any.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/numcodecs-0.15.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/bidict-0.23.1-py3-none-any.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/requests-2.34.2-py3-none-any.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-bas

In [8]:
# import sys
# import shutil
# import subprocess

# subprocess.run(
#     ["pip", "install", "--no-index", "--find-links", f"{ARTIFACTS}/wheels", "--no-deps", "donfig",
#      "tracksdata", "zarr>=3.0.10", "pyscipopt"],
#     check=True)

# shutil.copytree(f"{ARTIFACTS}/repo", REPO_DIR, dirs_exist_ok=True)
# shutil.copytree(f"{ARTIFACTS}/weights", f"{REPO_DIR}/weights", dirs_exist_ok=True)
# sys.path.insert(0, f"{REPO_DIR}/src")

# print("Weights:", os.listdir(f"{REPO_DIR}/weights/{METHOD}/split_0"))

## Inference on all test videos
Build a one-fold splits file listing every test video, then run prediction. PYTHONPATH=src makes the tracking_cellmot package importable without an internet install. Each video's tracks are exported as a .geff graph.

In [9]:
import json

test_stems = sorted(f[:-5] for f in os.listdir(TEST_DIR) if f.endswith(".zarr"))
print(f"{len(test_stems)} test videos")
print(f"One element in test_stems is {test_stems[0]}")

with open(f"{REPO_DIR}/kaggle_test_splits.json", "w") as f:
    json.dump([{"split": 0, "train": [], "test": test_stems}], f)

4 test videos
One element in test_stems is 44b6_0113de3b


In [10]:
#---------- Reading back the json file just created ------------
with open(f"{REPO_DIR}/kaggle_test_splits.json", "r") as f:
    json_file = json.load(f)

json_file

[{'split': 0,
  'train': [],
  'test': ['44b6_0113de3b',
   '44b6_0b24845f',
   '6bba_05b6850b',
   '6bba_05db0fb1']}]

In [11]:
import subprocess

# ----------------- command to run "predict_unet_transformer.py" which predicts data ------------------------
cmd = [
    "python", "scripts/predict_unet_transformer.py",
    "--data-dir", str(TEST_DIR), "--splits", "kaggle_test_splits.json", "--split", "0",
    "--weights", str(WEIGHTS), "--unet-batch-size", str(UNET_BATCH_SIZE),
    "--det-threshold", str(DET_THRESHOLD),
    "--ilp-edge-weight", str(ILP_EDGE_WEIGHT),
    "--ilp-appearance-weight", str(ILP_APPEARANCE_WEIGHT),
    "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE_WEIGHT),
    "--ilp-division-weight", str(ILP_DIVISION_WEIGHT),
]
if USE_ILP:
    cmd.append("--use-ilp")
if SLICE:
    cmd += ["--slice", SLICE]

print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": "src"}, check=True)

python scripts/predict_unet_transformer.py --data-dir /kaggle/input/competitions/biohub-cell-tracking-during-development/test --splits kaggle_test_splits.json --split 0 --weights weights/unet_transformer/split_0/edge_predictor_best.pth --unet-batch-size 4 --det-threshold 0.99 --ilp-edge-weight -1.0 --ilp-appearance-weight 0.1 --ilp-disappearance-weight 0.1 --ilp-division-weight 1.0 --use-ilp --slice :4


/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  from skimage.io import imread as sk_imread
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `reset_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  return _bootstrap._gcd_import(name[level:], package, level)


Fold 0: 4 datasets | weights=weights/unet_transformer/split_0/edge_predictor_best.pth | device=cuda | window_size=2 | pool_kernel_um=3.0
Saved 4 predictions to /kaggle/working/repo/predictions/unknown/unet_transformer/split_0


CompletedProcess(args=['python', 'scripts/predict_unet_transformer.py', '--data-dir', '/kaggle/input/competitions/biohub-cell-tracking-during-development/test', '--splits', 'kaggle_test_splits.json', '--split', '0', '--weights', 'weights/unet_transformer/split_0/edge_predictor_best.pth', '--unet-batch-size', '4', '--det-threshold', '0.99', '--ilp-edge-weight', '-1.0', '--ilp-appearance-weight', '0.1', '--ilp-disappearance-weight', '0.1', '--ilp-division-weight', '1.0', '--use-ilp', '--slice', ':4'], returncode=0)

# Build submission.csv
Flatten the predicted .geff graphs into the competition's CSV format: one node row per detection (t, z, y, x) and one edge row per link (source_id, target_id)

In [12]:
from pathlib import Path

import pandas as pd
import tracksdata as td

geffs = sorted(Path(REPO_DIR, "predictions").glob(f"*/{METHOD}/split_0/*.geff"))
print(f"{len(geffs)} prediction graphs")

rows = []
for g in geffs:
    name = g.stem
    graph = td.graph.IndexedRXGraph.from_geff(g)
    graph = graph[0] if isinstance(graph, tuple) else graph
    for r in graph.node_attrs().iter_rows(named=True):
        rows.append({
            "dataset": name, "row_type": "node", "node_id": int(r["node_id"]),
            "t": int(r["t"]), "z": int(round(r["z"])), "y": int(round(r["y"])),
            "x": int(round(r["x"])), "source_id": -1, "target_id": -1,
        })
    for r in graph.edge_attrs().iter_rows(named=True):
        rows.append({
            "dataset": name, "row_type": "edge", "node_id": -1,
            "t": -1, "z": -1, "y": -1, "x": -1,
            "source_id": int(r["source_id"]), "target_id": int(r["target_id"]),
        })

submission = pd.DataFrame(
    rows,
    columns=["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"],
)
submission.index.name = "id"
submission.to_csv("submission.csv")
print(f"Wrote submission.csv with {len(submission)} rows")
submission.head()

/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  from skimage.io import imread as sk_imread
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `reset_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  return _bootstrap._gcd_import(name[level:], package, level)


4 prediction graphs
Wrote submission.csv with 304792 rows


,dataset,row_type,node_id,t,z,y,x,source_id,target_id
id,,,,,,,,,
0,44b6_0113de3b,node,0,0,1,8,52,-1,-1
1,44b6_0113de3b,node,1,0,1,8,72,-1,-1
2,44b6_0113de3b,node,2,0,1,24,64,-1,-1
3,44b6_0113de3b,node,3,0,1,64,60,-1,-1
4,44b6_0113de3b,node,4,0,1,100,36,-1,-1


In [13]:
import shutil
import os

repo_path = REPO_DIR  # adjust if your folder is elsewhere

if os.path.exists(repo_path):
    shutil.rmtree(repo_path)
    print(f"Removed {repo_path}")
else:
    print("No repo folder found")

# sanity check: confirm only submission.csv remains
print(os.listdir('/kaggle/working'))

Removed /kaggle/working/repo
['submission.csv', '__notebook__.ipynb']


In [14]:
# import numpy as np
# import pandas as pd

In [15]:
# sample_sub = pd.read_csv('/kaggle/input/competitions/biohub-cell-tracking-during-development/sample_submission.csv')
# sample_sub

In [16]:
# submission_full = pd.read_csv("submission.csv")
# submission_full.head(20)

In [17]:
# sub = submission_full  # your full 4-dataset file

# print("="*60)
# print("A. UNEXPECTED VALUES / WHITESPACE")
# print("="*60)
# print("row_type values:", sub['row_type'].unique())
# print("dataset values match sample exactly?")
# print(sorted(sub['dataset'].unique()))
# print(sorted(sample['dataset'].unique()))
# # catch hidden whitespace
# print(sub['dataset'].apply(lambda s: s != s.strip()).sum(), "dataset values with stray whitespace")

# print("\n" + "="*60)
# print("B. SELF-LOOP / DEGENERATE EDGES")
# print("="*60)
# edges = sub[sub['row_type']=='edge']
# self_loops = edges[edges['source_id'] == edges['target_id']]
# print("self-loop edges (source_id == target_id):", len(self_loops))

# print("\n" + "="*60)
# print("C. DUPLICATE EDGES")
# print("="*60)
# dupe_edges = edges[edges.duplicated(subset=['dataset','source_id','target_id'], keep=False)]
# print("duplicate (dataset,source_id,target_id) edges:", len(dupe_edges))

# print("\n" + "="*60)
# print("D. TEMPORAL ORDERING OF EDGES (must go forward in time)")
# print("="*60)
# nodes = sub[sub['row_type']=='node']
# node_t = nodes.set_index(['dataset','node_id'])['t']
# bad_dir = 0
# for ds, grp in edges.groupby('dataset'):
#     t_src = grp.apply(lambda r: node_t.get((ds, r['source_id']), None), axis=1)
#     t_tgt = grp.apply(lambda r: node_t.get((ds, r['target_id']), None), axis=1)
#     bad = grp[(t_tgt.values <= t_src.values)]
#     if len(bad):
#         bad_dir += len(bad)
#         print(f"!! {ds}: {len(bad)} edges where target t <= source t")
# print("total bad-direction edges:", bad_dir)

# print("\n" + "="*60)
# print("E. MULTIPLE PARENTS (tree constraint — each node at most 1 incoming edge)")
# print("="*60)
# parent_counts = edges.groupby(['dataset','target_id']).size()
# multi_parent = parent_counts[parent_counts > 1]
# print("nodes with >1 incoming edge:", len(multi_parent))
# if len(multi_parent):
#     print(multi_parent.head(10))

# print("\n" + "="*60)
# print("F. FILE SIZE")
# print("="*60)
# import os
# # adjust path to your actual submitted file
# path = 'submission.csv'  # change if needed
# if os.path.exists(path):
#     print(f"{os.path.getsize(path)/1e6:.2f} MB")
# else:
#     print("point this at your actual submitted csv path")